In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 81 samples and 11899 columns.
First few rows of the data:


,New OS,vital.status NEW,New RemDur,event,Patient_ID,Cluster_Definition_Post_FAB_Correction,Batches,Platform-RNASeq,Platform-FM1,Platform-RPPA,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,84.00,1,21.14,1,3003486-N,Cluster_3,Batch1,RNASeq,FM1,RPPA,...,2.226803,1.146102,2.057132,1.723706,1.219951,1.308459,2.791821,2.582067,1.570612,1.833584
1,8.29,1,NaN,0,3044356-N,Cluster_2,Batch1,RNASeq,FM1,RPPA,...,4.098751,2.785802,4.125153,4.457178,2.766367,2.800134,4.732233,4.073847,2.629900,4.112465
2,22.43,1,NaN,0,3091926-N,Cluster_1,Batch1,RNASeq,FM1,RPPA,...,4.953971,3.637522,4.966192,5.198860,3.386643,3.627556,5.615686,4.996264,3.570893,5.019097
3,129.29,1,111.71,1,3103988-N,Cluster_2,Batch1,RNASeq,FM1,RPPA,...,3.943428,2.473518,3.655212,3.787883,2.247079,2.565096,4.512628,4.068320,2.731174,3.711038
4,36.57,1,NaN,0,3118298-N,Cluster_3,Batch1,RNASeq,FM1,RPPA,...,3.415620,1.849062,3.000872,2.935574,1.845778,1.992381,3.951988,3.377248,1.986683,3.110679


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for New OS or vital.status NEW: 0
Number of samples with complete data for both variables: 81
Number of censored samples (event_var == 0.0): 11
Number of events (event_var == 1.0): 70
Censored ratio (among complete): 11 / 81 = 0.136 (13.6%)
Number of samples with incomplete data for New RemDur or event: 46
Number of samples with complete data for both variables: 35
Number of censored samples (event_var == 0.0): 8
Number of events (event_var == 1.0): 27
Censored ratio (among complete): 8 / 35 = 0.229 (22.9%)
Skipping stats for EFS: columns not in preprocessed table ('Event.Free.Survival', 'unknown').
Updated stats for 2 of 3 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE165656/description.json
